In [ ]:
import re
import torch
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer
from transformers import AutoTokenizer, AutoModelForCausalLM


attribute_names = [
    'cleaning_service_quality',
    'order_packaging',
    'communication_and_responsiveness',
    'Driver_professionalism',
    'Service_speed'
]


for attr in attribute_names:

    # if df[attr] != 0: (the attribute is present in the review)

        # prompt model to predict the score for this review

        # if we fail to get the score in the response: reward = 10

# reward = average predicted score for each attribute - global score

# update the model
    



# 1. Setup Data
def prepare_dataset(df, attribute_names):
    dataset_dict = {"prompt": [], "global_score": [], "attribute": []}
    for _, row in df.iterrows():
        for attr in attribute_names:
            if row[attr] != 0:
                # We format the prompt exactly like your inference code
                prompt = (
                    "Task: Rate the sentiment of the specific Attribute mentioned in the Review.\n"
                    "Scale: 1 (Very Negative) to 9 (Very Positive).\n"
                    f"Review: {row['finalReview']}\n"
                    f"Attribute: {attr}\n"
                    "Score: <res>"
                )
                dataset_dict["prompt"].append(prompt)
                dataset_dict["global_score"].append(row['Satisfaction_final'])
                dataset_dict["attribute"].append(attr)
    return Dataset.from_dict(dataset_dict)

# 2. Reward Functions
def format_reward_func(completions, **kwargs) -> list[float]:
    """Rewards completions that follow the <res>DIGIT</res> format."""
    pattern = r"^\d</res>$" # Since the prompt ends with <res>, we expect 'score</res>'
    rewards = []
    for content in completions:
        if re.match(pattern, content.strip()):
            rewards.append(1.0)
        else:
            rewards.append(-2.0) # Heavy penalty for formatting
    return rewards

def consistency_reward_func(completions, global_score, **kwargs) -> list[float]:
    """Rewards scores that are closer to the global sentiment label."""
    rewards = []
    for content, target in zip(completions, global_score):
        match = re.search(r'\d', content)
        if match:
            predicted_score = int(match.group())
            # Calculate distance (closer to 0 is better)
            # We use a squared error or absolute difference
            diff = abs(predicted_score - target)
            reward = max(0, 1 - (diff / 4)) # Example: 1.0 if exact, 0.0 if > 4 away
            rewards.append(reward)
        else:
            rewards.append(0.0)
    return rewards

# 3. Training Script
model_id = "meta-llama/Llama-3.2-3B"

training_args = GRPOConfig(
    output_dir="llama-3.2-3b-sentiment-grpo",
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_generations=8, # Number of model responses to compare per prompt
    max_prompt_length=256,
    max_completion_length=10,
    save_steps=100,
    logging_steps=10,
    bf16=True,
)

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# Note: Using PEFT/LoRA is highly recommended for 3B models on consumer GPUs
trainer = GRPOTrainer(
    model=model_id,
    reward_funcs=[format_reward_func, consistency_reward_func],
    args=training_args,
    train_dataset=train_dataset, # Created from prepare_dataset
)

# trainer.train()